# GravityLab: N-Body Physics Analysis

This notebook serves as the analytical backend for the GravityLab project. While the WebGL application handles real-time visualization, this Python environment allows for:

1. **Algorithm Verification**: Implementing the physics logic using NumPy to verify energy conservation.
2. **Data Analysis**: Plotting trajectories and velocity distributions.
3. **High-Precision Simulation**: Running offline simulations with smaller time steps.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from mpl_toolkits.mplot3d import Axes3D

# Constants
G = 1.0
DT = 0.05
STEPS = 1000

## 1. Initialize Simulation

We initialize a "Galaxy" scenario similar to the web app: a massive central body surrounded by orbiting particles.

In [ ]:
def init_galaxy(n_particles=200):
    # Central Black Hole
    masses = np.ones(n_particles) * 1.0
    masses[0] = 1000.0 # Heavy center
    
    pos = np.zeros((n_particles, 3))
    vel = np.zeros((n_particles, 3))
    
    # Orbiting particles
    for i in range(1, n_particles):
        angle = np.random.rand() * 2 * np.pi
        dist = 10 + np.random.rand() * 40
        velocity = np.sqrt(G * masses[0] / dist)
        
        pos[i] = [np.cos(angle) * dist, (np.random.rand()-0.5)*2, np.sin(angle) * dist]
        vel[i] = [-np.sin(angle) * velocity, 0, np.cos(angle) * velocity]
        
    return pos, vel, masses

pos, vel, masses = init_galaxy()

## 2. Run Physics Loop (Vectorized)

We use a Brute Force $O(N^2)$ approach here for simplicity and exactness in verification, contrasting with the Barnes-Hut approximation in the web app.

In [ ]:
history = []

for step in range(STEPS):
    history.append(pos.copy())
    
    # Compute Forces (Brute Force Vectorized)
    # dx shape: (N, N, 3)
    dx = pos[:, np.newaxis, :] - pos[np.newaxis, :, :]
    
    # Distance squared
    r2 = np.sum(dx**2, axis=2)
    
    # Softening to avoid division by zero
    epsilon = 1.0
    r3 = (r2 + epsilon**2)**1.5
    
    # Force magnitude: G * m_j * m_i / r^3
    # We need acceleration, so divide by m_i. 
    # F_ij = G * m_i * m_j / r^3 * vec_r
    # a_i = Sum( G * m_j / r^3 * vec_r )
    
    # shape: (N, N)
    acc_matrix = (G * masses[np.newaxis, :] / r3)
    
    # Zero out self-interaction (diagonal is typically already handled by epsilon or r3, but explicit is good)
    np.fill_diagonal(acc_matrix, 0)
    
    # a shape: (N, 3) = sum((N, N, 1) * (N, N, 3))
    acc = np.sum(acc_matrix[:, :, np.newaxis] * dx, axis=1)
    
    # Semi-implicit Euler
    vel += acc * DT
    pos += vel * DT

history = np.array(history)

## 3. Visualization

Plotting the trajectories of the first 10 particles.

In [ ]:
fig = plt.figure(figsize=(10, 10))
ax = fig.add_subplot(111, projection='3d')

# Plot center body
ax.scatter(history[:, 0, 0], history[:, 0, 1], history[:, 0, 2], c='red', s=20, label='Black Hole')

# Plot orbits
for i in range(1, 20):
    ax.plot(history[:, i, 0], history[:, i, 1], history[:, i, 2], alpha=0.5)

ax.set_title('N-Body Simulation Trajectories')
ax.set_xlabel('X')
ax.set_ylabel('Y')
ax.set_zlabel('Z')
plt.legend()
plt.show()